# Automated Resume Screening & ML Fairness Audit Pipeline

This notebook runs the modular pipeline imported from the  package.


In [ ]:
# 1. Imports & Configuration
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import DATA_PATH, OUTPUT_DIR, RANDOM_STATE, LABEL_COL
from src.preprocessing import load_dataset, filter_demographics, combine_text_fields, extract_requirement_features
from src.feature_engineering import HybridFeatureTransformer, build_tfidf_vectorizer
from src.models import train_linear_svm, train_hybrid_classifier, train_rbf_svm, train_gaussian_nb, evaluate_model, cross_validate_model
from src.fairness import audit_demographic_fairness
from src.robustness import run_adversarial_attack_suite
from src.explainability import explain_candidate, get_top_features
from src.visualization import save_dashboard_summary, save_confusion_matrix_plot, save_top_features_plot, save_robustness_plot


In [ ]:
# 2. Load & Clean Dataset
df = load_dataset(DATA_PATH)
df = filter_demographics(df)
df["combined_text"] = combine_text_fields(df)
df = extract_requirement_features(df)
print("Cleaned dataset shape:", df.shape)
display(df[["name", "gender_group", "good_candidate", "weak_candidate", LABEL_COL]].head())


In [ ]:
# 3. Train / Test Split & Hybrid Feature Transformation
X_train_text, X_test_text, y_train, y_test, df_train, df_test = train_test_split(
    df["combined_text"],
    df[LABEL_COL],
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df[LABEL_COL],
)

hybrid_transformer = HybridFeatureTransformer()
X_train_hybrid = hybrid_transformer.fit_transform(X_train_text, df_train)
X_test_hybrid = hybrid_transformer.transform(X_test_text, df_test)
feature_names = hybrid_transformer.get_feature_names()
print("Hybrid Feature Matrix shape:", X_train_hybrid.shape)


In [ ]:
# 4. Model Training & Evaluation
hybrid_clf = train_hybrid_classifier(X_train_hybrid, y_train)
y_pred_hybrid = hybrid_clf.predict(X_test_hybrid)
metrics_hybrid = evaluate_model("Hybrid Classifier (Prod)", y_test, y_pred_hybrid, verbose=True)

cv_scores = cross_validate_model(train_hybrid_classifier(X_train_hybrid, y_train), X_train_hybrid, y_train, cv=5)
print(f"Cross-Validation F1: {cv_scores.mean():.3f} (± {cv_scores.std():.3f})")


In [ ]:
# 5. Demographic Fairness Audit
fairness_df = audit_demographic_fairness(df_test, y_test, y_pred_hybrid)
display(fairness_df)


In [ ]:
# 6. Adversarial Robustness Audit
tfidf_vec = build_tfidf_vectorizer().fit(X_train_text)
top_pos, _ = get_top_features(hybrid_clf.coef_[0], feature_names, top_k=10)
top_keywords = [f[0] for f in top_pos if f[0] in tfidf_vec.get_feature_names_out()]

robustness_df = run_adversarial_attack_suite(
    model=hybrid_clf,
    feature_transformer=hybrid_transformer,
    X_text_list=X_test_text.tolist(),
    df_test=df_test,
    top_keywords=top_keywords,
)
display(robustness_df)


In [ ]:
# 7. Single Candidate Local Explainer
explain_candidate(
    model=hybrid_clf,
    feature_transformer=hybrid_transformer,
    feature_names=feature_names,
    df=df_test,
    X_text=X_test_text,
    y_true=y_test,
    gender_series=df_test["gender_group"],
    fairness_df=fairness_df,
)
